In [1]:
import os
import requests

In [2]:
# Download the training text to local file
if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [4]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of character: {len(raw_text)}")
print(raw_text[:80])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow en


In [5]:
# Cleat test: use regex to split on punctuation and white-space
import re
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
# Clean training text
preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"Total number of words: {len(preprocessed)}")
print(preprocessed[:30])

Total number of words: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [7]:
# Compile sorted unique word list
all_tokens = sorted(list(set(preprocessed)))

# Add special tokens for end-of-text, unknown word
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab_size = len(all_tokens)
print(f"Total number of unique tokens: {vocab_size}")

Total number of unique tokens: 1132


In [8]:
# Create a vocabulary dictionary
vocab = { token:integer for integer,token in enumerate(all_tokens) }
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 30:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)


In [9]:
# Define vocabulary encoder/decoder
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Setup word to numeric dictionary
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        # Encode text into numeric vector
        preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]

        # Handle unknown words
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed ]

        # Convert words to numeric vector
        ids = [ self.str_to_int[s] for s in preprocessed ]
        return ids

    def decode(self, ids):
        # Decode numeric vector into text
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!\"()\'])', r'\1', text)
        return text

In [10]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "\"It's the last he painted, you know,\" Mrs. Gisburn said with pardonable pride."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [14]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [13]:
# Set standard tokenizer
from importlib.metadata import version
import tiktoken
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.12.0


In [16]:
# Test GPT-2 encoder
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text, allowed_special={ "<|endoftext|>" })
print(encoded_ids)

# Test GPT-2 decoder
decoded_text = tokenizer.decode(tokenizer.encode(text, allowed_special={ "<|endoftext|>" }))
print(decoded_text)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
